# DistilBERT extension: data and tokenization

Prepare the existing GoEmotions sentiment split for DistilBERT. This uses raw `text`, the same 680/120 train/validation split as the neural experiments, and leaves the held-out test split untouched until evaluation. Run from the repository root in Colab or another environment with the project files available.

In [0]:
%pip install -q "transformers>=4.45,<6.0" "torch>=2.4,<3.0" pandas scikit-learn

from pathlib import Path
import math
import sys

candidates = [Path.cwd(), *Path.cwd().parents]
ROOT_DIR = next((path for path in candidates if (path / 'src' / 'preprocessing.py').exists()), None)
if ROOT_DIR is None:
    raise FileNotFoundError('Start from the repository root or place the repository in the current directory.')
sys.path.insert(0, str(ROOT_DIR))

from src.preprocessing import SEED, load_splits, train_val_split
from src.models import build_transformer
from src.evaluate import compute_metrics, metrics_row, save_metrics
from transformers import AutoTokenizer
import numpy as np

MODEL_NAME = 'distilbert-base-uncased'
MAX_LENGTH_CAP = 64
SEED

Out[0]: 42


## Load the shared split

`load_splits` defaults to cleaned text, so pass `text_col='text'` to give DistilBERT the original comments. Only the training portion determines the validation split and token-length limit.

In [1]:
X_train_all, y_train_all, X_test, y_test = load_splits(text_col='text')
X_train, X_val, y_train, y_val = train_val_split(X_train_all, y_train_all, seed=SEED)
print(f'train={len(X_train)}, validation={len(X_val)}, test={len(X_test)}')
print(f'train labels={np.bincount(y_train)}, validation labels={np.bincount(y_val)}')
assert (len(X_train), len(X_val), len(X_test)) == (680, 120, 200)
assert set(np.unique(y_train_all)) == {0, 1}

train=680, validation=120, test=200
train labels=[340 340], validation labels=[60 60]


In [2]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
train_lengths = [len(tokenizer.encode(text, add_special_tokens=True)) for text in X_train]
p99_length = int(math.ceil(np.quantile(train_lengths, 0.99)))
MAX_LENGTH = MAX_LENGTH_CAP if p99_length <= MAX_LENGTH_CAP else p99_length
train_truncation_rate = float(np.mean(np.asarray(train_lengths) > MAX_LENGTH))
print(f'train subword lengths: p99={p99_length}, max={max(train_lengths)}')
print(f'max_length={MAX_LENGTH}; training truncation rate={train_truncation_rate:.1%}')
if p99_length > MAX_LENGTH_CAP:
    print('The training p99 exceeds 64 tokens, so max_length was increased to cover the training p99.')

train subword lengths: p99=36, max=44
max_length=64; training truncation rate=0.0%


## Tokenize without fitting on test data

The pretrained tokenizer is fixed, so no vocabulary is learned from this dataset. The maximum sequence length is selected from training comments only. Test comments are tokenized with that fixed setting for later evaluation; they are not used to choose it.

In [3]:
def tokenize_texts(texts):
    return tokenizer(
        list(texts),
        truncation=True,
        padding='max_length',
        max_length=MAX_LENGTH,
        return_attention_mask=True,
    )

tokenized_train = tokenize_texts(X_train)
tokenized_val = tokenize_texts(X_val)
tokenized_test = tokenize_texts(X_test)
for name, batch in [('train', tokenized_train), ('validation', tokenized_val), ('test', tokenized_test)]:
    ids = np.asarray(batch['input_ids'])
    masks = np.asarray(batch['attention_mask'])
    assert ids.shape == masks.shape == (len(batch['input_ids']), MAX_LENGTH)
    print(f'{name}: input_ids={ids.shape}, attention_mask={masks.shape}')

train: input_ids=(680, 64), attention_mask=(680, 64)
validation: input_ids=(120, 64), attention_mask=(120, 64)
test: input_ids=(200, 64), attention_mask=(200, 64)


## Fine-tune DistilBERT (J2)

The Trainer evaluates once per epoch and restores the checkpoint with the highest validation macro F1. The held-out test split is used only after training is complete. Checkpoints are written to a temporary directory and are not added to the repository.

In [4]:
from tempfile import TemporaryDirectory
import torch
from torch.utils.data import Dataset
from transformers import Trainer, TrainingArguments, set_seed

class SentimentDataset(Dataset):
    def __init__(self, tokenized, labels):
        self.input_ids = tokenized['input_ids']
        self.attention_mask = tokenized['attention_mask']
        self.labels = np.asarray(labels, dtype=np.int64)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        return {
            'input_ids': torch.tensor(self.input_ids[index], dtype=torch.long),
            'attention_mask': torch.tensor(self.attention_mask[index], dtype=torch.long),
            'labels': torch.tensor(self.labels[index], dtype=torch.long),
        }

train_dataset = SentimentDataset(tokenized_train, y_train)
val_dataset = SentimentDataset(tokenized_val, y_val)
test_dataset = SentimentDataset(tokenized_test, y_test)

def trainer_metrics(eval_prediction):
    predictions = np.asarray(eval_prediction.predictions)
    labels = np.asarray(eval_prediction.label_ids)
    return compute_metrics(labels, predictions.argmax(axis=-1))

set_seed(SEED)
model = build_transformer(MODEL_NAME)
training_args_config = dict(
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    greater_is_better=True,
    save_total_limit=1,
    logging_strategy='epoch',
    report_to='none',
    seed=SEED,
    data_seed=SEED,
    fp16=torch.cuda.is_available(),
)

with TemporaryDirectory(prefix='distilbert-j2-') as checkpoint_dir:
    training_args = TrainingArguments(output_dir=checkpoint_dir, **training_args_config)
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=trainer_metrics,
    )
    train_result = trainer.train()
    train_predictions = trainer.predict(train_dataset, metric_key_prefix='train')
    test_predictions = trainer.predict(test_dataset, metric_key_prefix='test')
    best_metric = float(trainer.state.best_metric)
    best_epoch = next(
        (entry['epoch'] for entry in trainer.state.log_history
         if entry.get('eval_f1_macro') == best_metric),
        None,
    )

train_time_s = float(train_result.metrics['train_runtime'])
n_params = sum(parameter.numel() for parameter in trainer.model.parameters())
rows = [
    metrics_row('distilbert', 'train', SEED, y_train, train_predictions.predictions.argmax(axis=-1), train_time_s, n_params),
    metrics_row('distilbert', 'test', SEED, y_test, test_predictions.predictions.argmax(axis=-1), train_time_s, n_params),
]
metrics_path = ROOT_DIR / 'outputs' / 'metrics' / 'distilbert.csv'
distilbert_metrics = save_metrics(rows, metrics_path)
print(f'Best validation macro F1: {best_metric:.4f} at epoch {best_epoch}')
print(f'Training time: {train_time_s:.1f}s; parameters: {n_params:,}')
display(distilbert_metrics)

{'loss': '0.661', 'grad_norm': '6.788', 'learning_rate': '1.349e-05', 'epoch': '1'}
{'eval_loss': '0.5577', 'eval_accuracy': '0.75', 'eval_precision': '0.6974', 'eval_recall': '0.8833', 'eval_f1': '0.7794', 'eval_f1_macro': '0.7455', 'eval_kappa': '0.5', 'eval_runtime': '6.381', 'eval_samples_per_second': '18.81', 'eval_steps_per_second': '1.254', 'epoch': '1'}
{'loss': '0.4096', 'grad_norm': '8.645', 'learning_rate': '6.822e-06', 'epoch': '2'}
{'eval_loss': '0.3784', 'eval_accuracy': '0.8583', 'eval_precision': '0.8772', 'eval_recall': '0.8333', 'eval_f1': '0.8547', 'eval_f1_macro': '0.8582', 'eval_kappa': '0.7167', 'eval_runtime': '5.788', 'eval_samples_per_second': '20.73', 'eval_steps_per_second': '1.382', 'epoch': '2'}
{'loss': '0.2577', 'grad_norm': '2.691', 'learning_rate': '1.55e-07', 'epoch': '3'}
{'eval_loss': '0.3728', 'eval_accuracy': '0.8667', 'eval_precision': '0.8667', 'eval_recall': '0.8667', 'eval_f1': '0.8667', 'eval_f1_macro': '0.8667', 'eval_kappa': '0.7333', 'eval_

  0%|          | 0/129 [00:00<?, ?it/s]C:\Users\jacob\Downloads\Suavitel\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
 88%|########7 | 7/8 [00:05<00:00,  1.27it/s]
                                                
100%|##########| 8/8 [00:05<00:00,  1.46it/s]
                                             
Writing model shards: 100%|##########| 1/1 [00:00<00:00,  2.58it/s]
C:\Users\jacob\Downloads\Suavitel\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
 88%|########7 | 7/8 [00:04<00:00,  1.38it/s]
                                                
100%|##########| 8/8 [00:05<00:00,  1.61it/s]
                                             
Writing model shards: 100%|##########

,model,split,seed,accuracy,precision,recall,f1,f1_macro,kappa,train_time_s,n_params
0,distilbert,train,42,0.935294,0.945783,0.923529,0.934524,0.935285,0.870588,418.5268,66955010
1,distilbert,test,42,0.840000,0.869565,0.800000,0.833333,0.839744,0.680000,418.5268,66955010


## Learning-rate comparison and three-seed evaluation (J3)

Compare `2e-5` and `5e-5` on validation using seed 42, then choose the learning rate with the higher best validation macro F1. After selection, retrain and evaluate seeds 42, 43, and 44. Test metrics are not used for choosing the rate. The validation search is saved separately from the shared model-metrics CSV.

In [5]:
import pandas as pd
from src.evaluate import load_all_metrics, summarize

def fit_distilbert(learning_rate, seed, evaluate_test=False, return_model=False):
    set_seed(seed)
    model = build_transformer(MODEL_NAME)
    with TemporaryDirectory(prefix=f'distilbert-{seed}-') as checkpoint_dir:
        args = TrainingArguments(
            output_dir=checkpoint_dir,
            learning_rate=learning_rate,
            per_device_train_batch_size=16,
            per_device_eval_batch_size=16,
            num_train_epochs=3,
            weight_decay=0.01,
            eval_strategy='epoch',
            save_strategy='epoch',
            load_best_model_at_end=True,
            metric_for_best_model='f1_macro',
            greater_is_better=True,
            save_total_limit=1,
            logging_strategy='epoch',
            report_to='none',
            seed=seed,
            data_seed=seed,
            fp16=torch.cuda.is_available(),
        )
        run = Trainer(
            model=model,
            args=args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
            compute_metrics=trainer_metrics,
        )
        train_output = run.train()
        best_score = float(run.state.best_metric)
        best_epoch = next(
            (entry['epoch'] for entry in run.state.log_history
             if entry.get('eval_f1_macro') == best_score),
            None,
        )
        result = {
            'learning_rate': learning_rate,
            'seed': seed,
            'best_val_f1_macro': best_score,
            'best_epoch': best_epoch,
            'train_time_s': float(train_output.metrics['train_runtime']),
            'n_params': sum(parameter.numel() for parameter in run.model.parameters()),
        }
        if evaluate_test:
            train_output = run.predict(train_dataset, metric_key_prefix='train')
            test_output = run.predict(test_dataset, metric_key_prefix='test')
            result['train_predictions'] = train_output.predictions.argmax(axis=-1)
            result['test_predictions'] = test_output.predictions.argmax(axis=-1)
        if return_model:
            result['model'] = run.model
    return result

learning_rates = [2e-5, 5e-5]
lr_trials = [fit_distilbert(rate, seed=SEED) for rate in learning_rates]
lr_results = pd.DataFrame(
    [{key: value for key, value in trial.items() if key != 'n_params'}
     for trial in lr_trials]
)
lr_results.to_csv(ROOT_DIR / 'outputs' / 'metrics' / 'distilbert_lr_search.csv', index=False)
display(lr_results.sort_values('best_val_f1_macro', ascending=False, ignore_index=True))
best_learning_rate = max(
    lr_trials, key=lambda trial: (trial['best_val_f1_macro'], -trial['learning_rate'])
)['learning_rate']
print(f'Selected learning rate from validation only: {best_learning_rate:g}')

seed_rows = []
for seed in [SEED, SEED + 1, SEED + 2]:
    result = fit_distilbert(
        best_learning_rate, seed=seed, evaluate_test=True, return_model=(seed == SEED)
    )
    if seed == SEED:
        attention_model = result.pop('model')
    seed_rows.extend([
        metrics_row('distilbert', 'train', seed, y_train, result['train_predictions'], result['train_time_s'], result['n_params']),
        metrics_row('distilbert', 'test', seed, y_test, result['test_predictions'], result['train_time_s'], result['n_params']),
    ])
    print(f"seed={seed}, val_macro_f1={result['best_val_f1_macro']:.4f}, best_epoch={result['best_epoch']}")

distilbert_metrics = save_metrics(
    seed_rows, ROOT_DIR / 'outputs' / 'metrics' / 'distilbert.csv'
)
distilbert_summary = summarize(
    load_all_metrics(ROOT_DIR / 'outputs' / 'metrics').query("model == 'distilbert'")
)
display(distilbert_summary)

# Keep only the seed-42 checkpoint in memory for the J4 attention figures.
del trainer, model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

{'loss': '0.661', 'grad_norm': '6.788', 'learning_rate': '1.349e-05', 'epoch': '1'}
{'eval_loss': '0.5577', 'eval_accuracy': '0.75', 'eval_precision': '0.6974', 'eval_recall': '0.8833', 'eval_f1': '0.7794', 'eval_f1_macro': '0.7455', 'eval_kappa': '0.5', 'eval_runtime': '6.079', 'eval_samples_per_second': '19.74', 'eval_steps_per_second': '1.316', 'epoch': '1'}
{'loss': '0.4096', 'grad_norm': '8.645', 'learning_rate': '6.822e-06', 'epoch': '2'}
{'eval_loss': '0.3784', 'eval_accuracy': '0.8583', 'eval_precision': '0.8772', 'eval_recall': '0.8333', 'eval_f1': '0.8547', 'eval_f1_macro': '0.8582', 'eval_kappa': '0.7167', 'eval_runtime': '6.091', 'eval_samples_per_second': '19.7', 'eval_steps_per_second': '1.313', 'epoch': '2'}
{'loss': '0.2577', 'grad_norm': '2.691', 'learning_rate': '1.55e-07', 'epoch': '3'}
{'eval_loss': '0.3728', 'eval_accuracy': '0.8667', 'eval_precision': '0.8667', 'eval_recall': '0.8667', 'eval_f1': '0.8667', 'eval_f1_macro': '0.8667', 'eval_kappa': '0.7333', 'eval_r

  0%|          | 0/129 [00:00<?, ?it/s]C:\Users\jacob\Downloads\Suavitel\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
100%|##########| 8/8 [00:05<00:00,  1.54it/s]
                                                
 33%|###3      | 43/129 [02:16<03:41,  2.58s/it]
                                             
Writing model shards: 100%|##########| 1/1 [00:01<00:00,  1.00s/it]
C:\Users\jacob\Downloads\Suavitel\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
100%|##########| 8/8 [00:05<00:00,  1.48it/s]
                                                
 67%|######6   | 86/129 [04:31<01:57,  2.73s/it]
                                             
Writing model shards: 100%|####

,learning_rate,seed,best_val_f1_macro,best_epoch,train_time_s
0,0.00002,42,0.866667,3.0,404.1162
1,0.00005,42,0.866630,2.0,397.1814


,model,split,n_runs,accuracy_mean,accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,f1_macro_mean,f1_macro_std,kappa_mean,kappa_std,train_time_s_mean,train_time_s_std,n_params_mean,n_params_std
0,distilbert,test,3,0.843333,0.005774,0.868464,0.020997,0.810000,0.017321,0.837941,0.004403,0.843117,0.005636,0.686667,0.011547,644.881133,424.074341,66955010.0,0.0
1,distilbert,train,3,0.935784,0.000849,0.951340,0.007409,0.918627,0.006123,0.934665,0.000425,0.935763,0.000833,0.871569,0.001698,644.881133,424.074341,66955010.0,0.0


## Inspect [CLS] attention on three test comments (J4)

For one comment each, inspect negation, possible irony, and an emoji. The model is the validation-selected seed-42 model from J3. Each plot averages the attention heads in the final layer for the `[CLS]` query. These weights show where the model routes attention; they are a diagnostic view, not a causal explanation of its prediction.

In [6]:
import matplotlib.pyplot as plt
from src.visualize import plot_cls_attention

if not hasattr(attention_model, 'set_attn_implementation'):
    raise RuntimeError('Upgrade transformers to a version that supports switching attention backends.')
attention_model.set_attn_implementation('eager')
attention_model.eval()
test_rows = pd.read_csv(ROOT_DIR / 'data' / 'processed' / 'test.csv', keep_default_na=False).set_index('comment_id')
attention_cases = {
    'negation': 'eesb4ga',
    'possible_irony': 'ef9cy2a',
    'emoji': 'ee5un57',
}
device = next(attention_model.parameters()).device

for category, comment_id in attention_cases.items():
    row = test_rows.loc[comment_id]
    encoded = tokenizer(
        str(row['text']),
        truncation=True,
        padding='max_length',
        max_length=MAX_LENGTH,
        return_attention_mask=True,
        return_tensors='pt',
    )
    encoded = {key: value.to(device) for key, value in encoded.items()}
    with torch.inference_mode():
        output = attention_model(**encoded, output_attentions=True, return_dict=True)
    if output.attentions is None:
        raise RuntimeError('The model did not return attention weights; check the eager attention backend.')
    visible_tokens = int(encoded['attention_mask'][0].sum().item())
    tokens = tokenizer.convert_ids_to_tokens(encoded['input_ids'][0, :visible_tokens].tolist())
    cls_weights = output.attentions[-1][0].float().mean(dim=0)[0, :visible_tokens].cpu().numpy()
    prediction = int(output.logits[0].argmax().item())
    class_names = ['negative', 'positive']
    print(f"{category} | id={comment_id} | true={row['sentiment']} | predicted={class_names[prediction]}")
    print(row['text'])
    fig = plot_cls_attention(
        tokens,
        cls_weights,
        title=f"{category.replace('_', ' ').title()} — CLS attention (last layer)",
        name=f'attention_cls_{category}',
    )
    display(fig)
    plt.close(fig)

negation | id=eesb4ga | true=positive | predicted=positive
I didn't find it too bad. I always favoured the 1500m and my wife is from Abu Dhabi
possible_irony | id=ef9cy2a | true=negative | predicted=negative
It’s more like a bunch of vaguely veiled racists and sexists and MAGAtards. Unless that’s what you consider “good.”
emoji | id=ee5un57 | true=negative | predicted=positive
I still gotta watch the last one. ☹


<Figure size 800x936 with 1 Axes>

<Figure size 800x1276 with 1 Axes>

<Figure size 800x494 with 1 Axes>

## Compare all five models (J6)

Use the same held-out 200 comments and select only the stratified Dummy baseline. The primary ranking is mean test macro F1; the table also shows train-test gaps, seed counts, fit time, parameter count, inference time, and interpretability notes. Inference timing covers model execution on already-prepared inputs, not text cleaning or tokenization. When a batch needs 1,000 rows, the 200 test comments are repeated in the same order. The ranking is descriptive and does not claim statistical significance.

In [15]:
import tensorflow as tf
from sklearn.dummy import DummyClassifier
from src.evaluate import comparison_table, load_all_metrics, measure_inference_time, summarize
from src.preprocessing import MAX_LEN, build_vectorizer, load_splits, vectorize
from src.visualize import plot_model_comparison

MODEL_ORDER = ['dummy_stratified', 'dense', 'rnn', 'lstm', 'distilbert']
all_metrics = load_all_metrics(ROOT_DIR / 'outputs' / 'metrics')
comparison_metrics = all_metrics[all_metrics['model'].isin(MODEL_ORDER)].copy()
missing_models = set(MODEL_ORDER) - set(comparison_metrics['model'])
if missing_models:
    raise RuntimeError(f'Missing model metrics: {sorted(missing_models)}. Run the required model notebooks first.')
distilbert_seeds = comparison_metrics.query("model == 'distilbert' and split == 'test'")['seed'].nunique()
if distilbert_seeds < 3:
    raise RuntimeError('J6 needs the three-seed DistilBERT results from J3; run the Transformer notebook through J3 first.')
for model_name in MODEL_ORDER:
    present_splits = set(comparison_metrics.loc[comparison_metrics['model'] == model_name, 'split'])
    if not {'train', 'test'}.issubset(present_splits):
        raise RuntimeError(f'{model_name} is missing train or test metrics.')

X_train_clean, y_train_clean, X_test_clean, y_test_clean = load_splits()
vectorizer = build_vectorizer(X_train_clean, max_len=MAX_LEN)
X_train_ids = vectorize(vectorizer, X_train_clean)
X_test_ids = vectorize(vectorizer, X_test_clean)

dummy = DummyClassifier(strategy='stratified', random_state=SEED).fit(X_train_ids, y_train_clean)
inference_seconds = {
    'dummy_stratified': measure_inference_time(
        lambda batch: dummy.predict(batch), X_test_ids, repeats=5
    )
}
keras_models = {}
for model_name in ['dense', 'rnn', 'lstm']:
    keras_model = tf.keras.models.load_model(
        ROOT_DIR / 'outputs' / 'saved_models' / f'{model_name}.keras', compile=False
    )
    keras_models[model_name] = keras_model
    inference_seconds[model_name] = measure_inference_time(
        lambda batch, model=keras_model: model.predict(batch, batch_size=32, verbose=0),
        X_test_ids,
        repeats=5,
    )

# Reuse the validation-selected seed-42 checkpoint kept in memory from J3.
attention_model.set_attn_implementation('sdpa')
attention_model.eval()
hf_device = next(attention_model.parameters()).device
hf_test_rows = list(zip(tokenized_test['input_ids'], tokenized_test['attention_mask']))

def predict_hf_rows(rows):
    batch_predictions = []
    for start in range(0, len(rows), 32):
        part = rows[start:start + 32]
        input_ids = torch.tensor([item[0] for item in part], dtype=torch.long, device=hf_device)
        attention_mask = torch.tensor([item[1] for item in part], dtype=torch.long, device=hf_device)
        with torch.inference_mode():
            logits = attention_model(input_ids=input_ids, attention_mask=attention_mask).logits
        batch_predictions.append(logits.argmax(dim=-1).cpu().numpy())
    return np.concatenate(batch_predictions)

inference_seconds['distilbert'] = measure_inference_time(
    predict_hf_rows, hf_test_rows, repeats=5
)

comparison = comparison_table(comparison_metrics)
comparison['inference_seconds_per_1000'] = comparison['model'].map(inference_seconds)
interpretability = {
    'dummy_stratified': 'Transparent random baseline; does not read text',
    'dense': 'Low; mean embeddings discard token order',
    'rnn': 'Limited; sequential hidden states are difficult to inspect',
    'lstm': 'Limited; gates add memory but remain difficult to inspect',
    'distilbert': 'Attention is inspectable as a diagnostic, not a causal explanation',
}
comparison['interpretability'] = comparison['model'].map(interpretability)
comparison['model'] = pd.Categorical(comparison['model'], MODEL_ORDER, ordered=True)
comparison = comparison.sort_values('model').reset_index(drop=True)
comparison['model'] = comparison['model'].astype(str)
comparison_path = ROOT_DIR / 'outputs' / 'metrics' / 'comparison.csv'
comparison.to_csv(comparison_path, index=False)

display(comparison[[
    'model', 'train_n_runs', 'test_n_runs',
    'train_f1_macro_mean', 'train_f1_macro_std',
    'test_f1_macro_mean', 'test_f1_macro_std', 'train_test_gap_f1_macro',
    'train_train_time_s_mean', 'train_train_time_s_std', 'train_n_params_mean', 'train_n_params_std',
    'inference_seconds_per_1000', 'interpretability',
]])

summary = summarize(comparison_metrics)
comparison_figure = plot_model_comparison(
    summary, metric='f1_macro', name='comparison', order=MODEL_ORDER
)
display(comparison_figure)
plt.close(comparison_figure)

ranked = comparison.sort_values('test_f1_macro_mean', ascending=False)
leader = ranked.iloc[0]
runner_up = ranked.iloc[1]
print('By mean test macro F1, {} ranks first ({:.3f} +/- {:.3f}; n={}). The runner-up is {} ({:.3f} +/- {:.3f}).'.format(
    leader.model, leader.test_f1_macro_mean, leader.test_f1_macro_std, int(leader.test_n_runs),
    runner_up.model, runner_up.test_f1_macro_mean, runner_up.test_f1_macro_std,
))
print('Use the overfit gap, inference time, parameter count, and interpretability notes above to qualify this quality-first ranking.')

By mean test macro F1, distilbert ranks first (0.843 +/- 0.006; n=3). The runner-up is dense (0.741 +/- 0.043).


,model,train_n_runs,test_n_runs,train_f1_macro_mean,train_f1_macro_std,test_f1_macro_mean,test_f1_macro_std,train_test_gap_f1_macro,train_train_time_s_mean,train_train_time_s_std,train_n_params_mean,train_n_params_std,inference_seconds_per_1000,interpretability
0,dummy_stratified,5,5,0.497826,0.026648,0.516403,0.025010,-0.018576,0.000159,0.000033,0.0,0.0,0.000458,Transparent random baseline; does not read text
1,dense,5,5,0.990249,0.006399,0.740799,0.043189,0.249450,0.855176,0.057807,76801.0,0.0,0.183573,Low; mean embeddings discard token order
2,rnn,5,5,0.812023,0.107392,0.550729,0.046952,0.261294,1.301004,0.108169,81985.0,0.0,0.184841,Limited; sequential hidden states are difficul...
3,lstm,5,5,0.945198,0.041232,0.700592,0.021269,0.244606,1.505595,0.121852,163873.0,0.0,0.221398,Limited; gates add memory but remain difficult...
4,distilbert,3,3,0.935763,0.000833,0.843117,0.005636,0.092646,644.881133,424.074341,66955010.0,0.0,47.193307,"Attention is inspectable as a diagnostic, not ..."


<Figure size 850x400 with 1 Axes>